# BDCAgent

## Requirements

In [38]:
from langchain.chat_models import ChatOpenAI
from typing import List, Dict
from langchain.tools import tool
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import io
import sys
from typing import Any, Dict, Optional, Union
from dataclasses import dataclass
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## Planning AI

In [40]:
class PlanningAgent:   
    def __init__(self, agent: ChatOpenAI):
        self.agent = agent

    def predict(self, prompt: str):
        return self.agent.predict(prompt)

    def send_message_block(self, new_message_block: Dict, messages_history: List[Dict] = None):
        user_input = new_message_block["content"][0]["text"]
        
        try:
            format_prompt = """Format this request into the following structure exactly:
            If it's a cleaning request, use this format ONLY:
            I need to clean the data
            Action: CleanData
            Action Input: [file_path]

            If it's a correlation request, use this format ONLY:
            I need to find the correlation
            Action: FindCorrelation
            Action Input: [file_path] [column1,column2]

            Format ONLY ONE of these based on the request.
            Request: """
            
            formatted_response = self.predict(format_prompt + user_input)
                        
            return messages_history, formatted_response
            
        except Exception as e:
            print(f"Error: {e}")
            return messages_history, formatted_response

    def send_message(self, new_message: str, messages_history: List[Dict] = None):
        """Send a new message to the agent."""
        if messages_history is None:
            messages_history = []
            
        new_message_block = {
            "role": "user",
            "content": [{"type": "text", "text": new_message}]
        }

        return self.send_message_block(new_message_block=new_message_block, messages_history=messages_history)

## Cleaning AI

In [41]:
@tool
def clean_dates_tool(file_path: str) -> str:
    """Converts date columns to datetime format"""
    try:
        df = pd.read_csv(file_path)
        date_columns = df.columns[df.columns.str.contains('date', case=False)]
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        df.to_csv("cleaned_data/cleaned_data.csv", index=False)
        return f"Dates cleaned in columns: {list(date_columns)}"
    except Exception as e:
        return f"Error cleaning dates: {str(e)}"

@tool
def remove_nulls_tool(file_path: str) -> str:
    """Removes rows with null values"""
    try:
        df = pd.read_csv(file_path)
        initial_rows = len(df)
        df = df.dropna()
        df.to_csv("cleaned_data/cleaned_data.csv", index=False)
        return f"Removed {initial_rows - len(df)} rows with null values"
    except Exception as e:
        return f"Error removing nulls: {str(e)}"

@tool
def fix_dtypes_tool(file_path: str) -> str:
    """Converts columns to appropriate data types"""
    try:
        df = pd.read_csv(file_path)
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    df[col] = pd.to_numeric(df[col], errors='raise')
                except:
                    continue
        df.to_csv('cleaned_data/cleaned_data.csv', index=False)
        return f"Fixed data types for numeric columns"
    except Exception as e:
        return f"Error fixing data types: {str(e)}"

class CleaningAgent:
    def __init__(self, agent: ChatOpenAI):
        self.agent = agent
        
    def clean_data(self, formatted_instruction: str):
        try:
            # Extract the file path
            file_path = formatted_instruction.split("Action Input:")[1].strip()
            
            if not os.path.exists("cleaned_data"):
                os.mkdir("cleaned_data")

            # Run all cleaning tools in sequence
            results = []
            results.append(clean_dates_tool(file_path))

            # after first cleaning
            file_path = r"cleanleaned_data/cleaned_data.csv"
            results.append(remove_nulls_tool(file_path))
            results.append(fix_dtypes_tool(file_path))
            
            return "\n".join(results)
            
        except Exception as e:
            print(f"Error cleaning data: {e}")
            return None

## Causation AI

In [42]:


@tool
def correlation_tool(params: str) -> str:
    """Calculate correlation between two variables"""
    try:
        file_path = params.split('[')[0].strip()
        vars_str = params.split('[')[1].split(']')[0]
        var1, var2 = [v.strip() for v in vars_str.split(',')]
        
        df = pd.read_csv(file_path)
        correlation = df[var1].corr(df[var2])
        return f"Correlation between {var1} and {var2}: {correlation:.4f}"
    except Exception as e:
        return f"Error calculating correlation: {str(e)}"

@tool
def granger_causality_tool(params: str) -> str:
    """Test for Granger causality between two variables"""
    try:
        file_path = params.split('[')[0].strip()
        vars_str = params.split('[')[1].split(']')[0]
        var1, var2 = [v.strip() for v in vars_str.split(',')]
        
        df = pd.read_csv(file_path)
        data = pd.DataFrame({var1: df[var1], var2: df[var2]})
        
        
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        
        try:
            result = grangercausalitytests(data, maxlag=5)
            detailed_results = []
            
            for lag in range(1, 6):
                test_results = result[lag][0]
                detailed_results.append(
                    f"Lag {lag}:\n"
                    f"F-test: F={test_results['ssr_ftest'][0]:.4f}, p={test_results['ssr_ftest'][1]:.4f}\n"
                    f"Chi-squared test: chi2={test_results['ssr_chi2test'][0]:.4f}, p={test_results['ssr_chi2test'][1]:.4f}\n"
                    f"Likelihood ratio test: chi2={test_results['lrtest'][0]:.4f}, p={test_results['lrtest'][1]:.4f}\n"
                    f"Parameter F-test: F={test_results['params_ftest'][0]:.4f}, p={test_results['params_ftest'][1]:.4f}\n"
                )
        finally:
            sys.stdout = old_stdout
        
        return "\n".join(detailed_results)
    except Exception as e:
        return f"Error testing Granger causality: {str(e)}"

class CausationAgent:
    def __init__(self, agent: ChatOpenAI):
        self.agent = agent
        
    def analyze_causation(self, formatted_instruction: str):
        try:
            # Extract file path and variables from instruction
            params = formatted_instruction.split("Action Input:")[1].strip()
            
            # Run analysis tools
            results = []
            results.append(correlation_tool(params))
            results.append(granger_causality_tool(params))
            
            return "\n".join(results)
            
        except Exception as e:
            print(f"Error analyzing causation: {e}")
            return None

In [43]:


@dataclass
class InterpretationResult:
    summary: str
    insights: list[str]
    recommendations: list[str]
    confidence_score: float
    timestamp: datetime
    metadata: Dict[str, Any]

class InterpretationAgent:
    def __init__(self, agent: Any, confidence_threshold: float = 0.7):
        """
        Initialize the InterpretationAgent with an LLM agent and confidence threshold.
        
        Args:
            agent: The language model agent to use for interpretation
            confidence_threshold: Minimum confidence score to consider an interpretation valid
        """
        self.agent = agent
        self.confidence_threshold = confidence_threshold
        self.interpretation_history = []
    
    def interpret(self, prompt: str) -> str:
        """
        Get raw interpretation from the agent.
        
        Args:
            prompt: The prompt to interpret
            
        Returns:
            Raw interpretation response
        """
        try:
            return self.agent.predict(prompt)
        except Exception as e:
            raise Exception(f"Error during interpretation: {e}")
    
    def interpret_results(self, formatted_instruction: str):
        try:
            # Parse correlation and Granger results
            correlation_line = formatted_instruction.split('\n')[0]
            correlation_value = float(correlation_line.split(': ')[1])
            
            # Create interpretation
            interpretation = []
            
            # Interpret correlation
            interpretation.append(f"The correlation coefficient of {correlation_value:.4f} indicates:")
            if abs(correlation_value) > 0.8:
                interpretation.append("- A very strong relationship between Open and Close prices")
            elif abs(correlation_value) > 0.6:
                interpretation.append("- A strong relationship between Open and Close prices")
            elif abs(correlation_value) > 0.4:
                interpretation.append("- A moderate relationship between Open and Close prices")
            else:
                interpretation.append("- A weak relationship between Open and Close prices")
                
            # Interpret Granger causality
            if "Granger" in formatted_instruction:
                lag_results = formatted_instruction.split('Lag')
                significant_lags = []
                
                for lag_result in lag_results[1:]:  # Skip first split which is correlation
                    if 'p=0.0000' in lag_result or 'p=0.00' in lag_result:
                        lag_num = lag_result.split(':')[0].strip()
                        significant_lags.append(lag_num)
                
                if significant_lags:
                    interpretation.append(f"\nGranger causality tests show significant relationships at lags: {', '.join(significant_lags)}")
                    interpretation.append("This suggests that past Open prices help predict Close prices")
            
            return "\n".join(interpretation)
            
        except Exception as e:
            print(f"Error analyzing interpretation: {e}")
            return None
    
    def _extract_summary(self, raw_interpretation: str) -> str:
        """Extract key summary from raw interpretation."""
        # Implementation would depend on the expected format of raw_interpretation
        # This is a placeholder implementation
        return raw_interpretation.split('\n')[0] if raw_interpretation else ""
    
    def _extract_insights(self, raw_interpretation: str) -> list[str]:
        """Extract key insights from raw interpretation."""
        # Placeholder implementation
        insights = []
        lines = raw_interpretation.split('\n')
        for line in lines:
            if line.startswith('- '):
                insights.append(line[2:])
        return insights
    
    def _extract_recommendations(self, raw_interpretation: str) -> list[str]:
        """Extract actionable recommendations from raw interpretation."""
        # Placeholder implementation
        recommendations = []
        lines = raw_interpretation.split('\n')
        in_recommendations = False
        for line in lines:
            if 'recommendations:' in line.lower():
                in_recommendations = True
                continue
            if in_recommendations and line.strip():
                recommendations.append(line.strip())
        return recommendations
    
    def _calculate_confidence(self, 
                            interpretation: str, 
                            data: Optional[Union[pd.DataFrame, Dict[str, Any]]] = None
                            ) -> float:
        """
        Calculate confidence score for the interpretation.
        
        This could be based on factors like:
        - Consistency with historical interpretations
        - Data quality metrics
        - LLM confidence scores
        - Presence of key expected elements
        """
        # Placeholder implementation
        base_score = 0.8  # Default confidence
        
        # Adjust based on interpretation length
        if len(interpretation) < 50:
            base_score -= 0.2
            
        # Adjust based on data presence
        if data is not None:
            base_score += 0.1
            
        return min(1.0, max(0.0, base_score))
    
    def _validate_interpretation(self, result: InterpretationResult) -> bool:
        """Validate interpretation result meets quality standards."""
        if result.confidence_score < self.confidence_threshold:
            return False
        if not result.summary or not result.insights:
            return False
        return True
    
    def _get_data_shape(self, data: Optional[Union[pd.DataFrame, Dict[str, Any]]]) -> Optional[tuple]:
        """Get shape of input data if available."""
        if isinstance(data, pd.DataFrame):
            return data.shape
        elif isinstance(data, dict):
            return (len(data),)
        return None
    
    def get_interpretation_history(self) -> list[InterpretationResult]:
        """Get history of all successful interpretations."""
        return self.interpretation_history

# Main

In [44]:
def prompt(message, file_path):
    # define agents, planning agent should dynamically adjust which agentss are needed
    planning_agent = PlanningAgent(agent=llm)
    cleaning_agent = CleaningAgent(agent=llm)
    causation_agent = CausationAgent(agent=llm)
    interpretation_agent = InterpretationAgent(agent=llm)

    agent_calls = {
        "CleanData": cleaning_agent.clean_data,
        "FindCorrelation": causation_agent.analyze_causation,
        "Interpret": interpretation_agent.interpret_results
    }

    # split the message into tasks, this needs to be improved
    messages = ["I need to clean the data", "I need to find the correlation between Open and Close", "Interpret"]

    for message in messages:
        print(f"User Message: {message}")
        message_history, response = planning_agent.send_message(message)
        formatted_response = response.replace("[file_path]", file_path)
        #print(f"Planning Agent Response:\n{formatted_response}\n")

        if message == "Interpret":
            print("Interpretation Agent Response:")
            #print(result)
            interpretation = agent_calls["Interpret"](result)
            return interpretation

        if "Action:" in formatted_response:
            action = formatted_response.split("Action:")[1].split("\n")[0].strip()
            if action in agent_calls:
                result = agent_calls[action](formatted_response)
        
    return result
        

In [45]:
def main(message, file_path):
    df = pd.read_csv(file_path)
    display(df.head())

    #print("---")

    final_result = prompt(message, file_path)
    print(final_result)

main(message = "I need to clean the data and I need to find the correlation", file_path = r"C:\Datasets\stocks\stocks\A.csv")

,Date,Open,High,Low,Close,Adj Close,Volume
0,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300
1,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100
2,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800
3,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600
4,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200


User Message: I need to clean the data
User Message: I need to find the correlation between Open and Close
User Message: Interpret
Interpretation Agent Response:
The correlation coefficient of 0.9986 indicates:
- A very strong relationship between Open and Close prices
